# Dynamic Social Media Networks

In this notebook, we analyse temporal changes in dynamic networks generated from Twitter/X data. Specifically, we will examine a **dynamic follower network** to understand how social media relationships evolve over time.

As part of this analysis, we will explore how to generate time series data by applying network measures to each temporal window in a dynamic network. This allows us to quantify and visualise changes in network structure across different time periods.

In [ ]:
from datetime import datetime
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.float_format', '{:,.3f}'.format)

## Data Processing

We will work with a sample file containing Twitter/X follower data (i.e. user *A* follows user *B*), accompanied by timestamp information indicating when each following relationship was established. This temporal information allows us to reconstruct the evolution of the follower network over time.

In [ ]:
# note that we tell Pandas to parse the date value in the "date" column.
df1 = pd.read_csv("dynamic-friendships.csv", sep="\t", parse_dates=["date"])
df1.head(10)

Let's examine the range of values in the date column to understand the temporal scope of our dataset:

In [ ]:
df1["date"].min(), df1["date"].max()

Based on the temporal range of the data, we will divide the dataset into yearly time windows spanning 2016 to 2019. This annual segmentation provides enough granularity to observe meaningful changes while maintaining adequate data in each window.

In [ ]:
years = list(range(2016, 2020))

In [ ]:
window_frames = {}
for year in years:
    win_start = datetime(year, 1, 1)
    win_end = datetime(year, 12, 31)
    window_frames[year] = df1[(df1["date"]>=win_start) & (df1["date"]<=win_end)]
    print(f"Window {year}: Date >= {win_start.date()} and < {win_end.date()} -> {len(window_frames[year])} follows")

## Time Window Network Construction

From these window frames, we can create a directed network for each time window using the `nx.from_pandas_edgelist()` function with `nx.DiGraph()` to preserve the directional nature of follower relationships:

In [ ]:
window_networks = {}
for year in years:
    # create a directed network from the corresponding DataFrame
    g = nx.from_pandas_edgelist(window_frames[year], "follower", "followee", create_using=nx.DiGraph())  
    print(f"Window {year} network has {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
    window_networks[year] = g

We can visualise the different window networks to observe structural evolution over time. Note that these plots only display the following relationships that were established in each respective year.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 13))
row, col = 0, 0
for i, year in enumerate(years):
    pos = nx.circular_layout(window_networks[year])
    # draw the network as a subplot
    nx.draw_networkx(window_networks[year], pos, 
                     ax=ax[row, col],
                     with_labels=True, font_size=11, node_size=800, node_color="lightblue")
    ax[row, col].set_title(f"Network {year}", fontsize=12)
    ax[row, col].set_axis_off()
    col += 1
    if col == 2:
        col = 0
        row += 1
plt.show()

We can characterise each network and view these measures as a time series, representing a sequence of values measured at different temporal points. In this context, each point corresponds to our defined time windows.

For instance, we can compute the `density` and `reciprocity` metrics for each time window. Density measures how well connected the network is, whilst reciprocity quantifies the proportion of mutual following relationships:

In [ ]:
rows = []
for year in years:
    row = {"Year": year, 
           "Density": nx.density(window_networks[year]), 
           "Reciprocity": nx.reciprocity(window_networks[year])} 
    rows.append(row)
df_stats1 = pd.DataFrame(rows).set_index("Year")
df_stats1

We can plot these temporal changes to visualise how network characteristics evolve over time:

In [ ]:
ax = df_stats1.plot(figsize=(9, 5), style=".-", ms=12, fontsize=12, zorder=3)
# specify labels for the axes
ax.set_xlabel("Time Window (Year)", fontsize=12)
ax.set_ylabel("Measure Value", fontsize=12)
ax.legend(fontsize=12)
# adjust the axes
ax.set_xticks(years)
ax.set_ylim(0, 0.4)
ax.set_xlim(years[0], years[-1])
ax.xaxis.grid()
plt.show()